# 04 — Analisis tambahan untuk tanggapan reviewer (JIEE 1757)
Notebook ini mereproduksi seluruh analisis baru pada naskah revisi. Fungsi bersama identik dengan notebook 01–03. Jalankan dari folder `notebooks/` repositori (dataset di `data/`, hasil notebook 01–03 di `results/`) atau dari folder kerja yang berisi `healthcare-dataset-stroke-data.csv` dan `hasil_notebook/`. Hasil disimpan di `results/revisi1/`.

| Bagian | Komentar reviewer |
|---|---|
| 1. Fungsi bersama | — |
| 2. Nested CV + ambang Youden + indikator BMI | #5, #11, #19, #20 |
| 3. Ringkasan nested CV | #5, #11, #19, #20 |
| 4. Selisih per fold, uji primer Nadeau–Bengio, stabilitas peringkat | #12, #18, #21, #29 |
| 5. BMI kosong, skenario prevalensi dan ROS, jarak, 20 seed, usia saja, model akhir, PI | #5, #7, #9, #13, #15, #16, #25, #27 |
| 6. Gambar kalibrasi revisi | #23 |
| 7. Verifikasi imbalanced-learn (versi lengkap: notebook 05) | #28 |

## 1. Fungsi bersama

In [ ]:
# Fungsi bersama, identik dengan notebook 01/02 (stroke_leakage_JIEE / stroke_revisi_M1_M3)
import os, json, time, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV, train_test_split, cross_val_predict
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, recall_score,
                             precision_score, f1_score, brier_score_loss, roc_curve, confusion_matrix)
SEED = 42; N_REPEATS = 2; N_ITER = 12; N_BOOT = 1000
HERE = os.getcwd()
def _cari(*opsi):
    for o in opsi:
        if os.path.exists(o): return os.path.abspath(o)
    raise FileNotFoundError(f"Tidak ditemukan: {opsi}")
# Bekerja dari folder notebooks/ di repositori GitHub maupun dari folder kerja yang berisi dataset
DATA = _cari("healthcare-dataset-stroke-data.csv", "../data/healthcare-dataset-stroke-data.csv", "data/healthcare-dataset-stroke-data.csv")
HASIL = _cari("hasil_notebook", "../results", "results")
OUTD = os.path.join(HASIL, "revisi1"); os.makedirs(OUTD, exist_ok=True)
df = pd.read_csv(DATA).drop(columns="id")
df = df[df.gender != "Other"].reset_index(drop=True)
y = df.pop("stroke").values; X = df.copy()
X["bmi_kosong"] = X.bmi.isna().astype(int)          # hanya dipakai pada varian indikator
NUM = ["age", "avg_glucose_level", "bmi"]; BIN = ["hypertension", "heart_disease"]
CAT = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
CATS = [sorted(X[c].unique()) for c in CAT]
def make_prep(indicator=False):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
        ("bin", "passthrough", BIN + (["bmi_kosong"] if indicator else [])),
        ("cat", OneHotEncoder(categories=CATS, drop="first", handle_unknown="ignore", sparse_output=False), CAT)])
NUM_IDX = list(range(len(NUM))); GROUPS = [[len(NUM)], [len(NUM) + 1]]; _s = len(NUM) + 2
for c in CATS: GROUPS.append(list(range(_s, _s + len(c) - 1))); _s += len(c) - 1
def smote(X, y, k=5, seed=0):
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xm))).fit(Xm)
    idx = nn.kneighbors(Xm, return_distance=False)[:, 1:]
    i = r.randint(0, len(Xm), n); j = idx[i, r.randint(0, idx.shape[1], n)]; g = r.rand(n, 1)
    return np.vstack([X, Xm[i] + g * (Xm[j] - Xm[i])]), np.r_[y, np.ones(n, int)]
def rus(X, y, seed=0):
    r = np.random.RandomState(seed); pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    keep = np.r_[pos, r.choice(neg, len(pos), replace=False)]; return X[keep], y[keep]
def ros(X, y, seed=0):
    r = np.random.RandomState(seed); pos = np.where(y == 1)[0]; n = int((y == 0).sum() - len(pos))
    add = r.choice(pos, n, replace=True); return np.vstack([X, X[add]]), np.r_[y, np.ones(n, int)]
def smotenc(X, y, k=5, seed=0):
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    med_sd = np.median(Xm[:, NUM_IDX].std(axis=0))
    dn = ((Xm[:, None, NUM_IDX] - Xm[None, :, NUM_IDX]) ** 2).sum(-1)
    dc = sum((np.abs(Xm[:, None, g] - Xm[None, :, g]).sum(-1) > 0).astype(float) for g in GROUPS)
    D = np.sqrt(dn + dc * med_sd ** 2); np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]
    i = r.randint(0, len(Xm), n); j = nn[i, r.randint(0, k, n)]; gap = r.rand(n, 1)
    S = Xm[i].copy(); S[:, NUM_IDX] = Xm[i][:, NUM_IDX] + gap * (Xm[j][:, NUM_IDX] - Xm[i][:, NUM_IDX])
    for g in GROUPS:
        neigh = Xm[:, g][nn[i]]
        for t in range(n):
            vals, cnt = np.unique(neigh[t], axis=0, return_counts=True); S[t, g] = vals[cnt.argmax()]
    return np.vstack([X, S]), np.r_[y, np.ones(n, int)]
class Resampled(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator=None, sampler="none", random_state=0):
        self.estimator = estimator; self.sampler = sampler; self.random_state = random_state
    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        f = {"smote": smote, "rus": rus, "smotenc": smotenc, "ros": ros}.get(self.sampler)
        if f: X, y = f(X, y, seed=self.random_state)
        self.est_ = clone(self.estimator).fit(X, y); self.classes_ = self.est_.classes_; return self
    def predict_proba(self, X): return self.est_.predict_proba(np.asarray(X, float))
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)
def models(cw=False):
    w = "balanced" if cw else None
    return {"LR": (LogisticRegression(max_iter=3000, class_weight=w), {"C": list(np.logspace(-3, 2, 30))}),
            "RF": (RandomForestClassifier(n_estimators=200, class_weight=w, n_jobs=-1, random_state=SEED),
                   {"max_depth": [3, 5, 8, 12, None], "min_samples_leaf": [1, 5, 10, 20, 50], "max_features": ["sqrt", 0.5, 0.8]}),
            "HGB": (HistGradientBoostingClassifier(class_weight=w, random_state=SEED),
                    {"learning_rate": list(np.logspace(-2.5, -0.5, 20)), "max_depth": [2, 3, 4, None], "max_leaf_nodes": [7, 15, 31],
                     "min_samples_leaf": [20, 50, 100], "l2_regularization": list(np.logspace(-3, 1, 10))})}
def metrics(yt, p, thr=0.5):
    yh = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    return dict(Accuracy=accuracy_score(yt, yh), ROC_AUC=roc_auc_score(yt, p), PR_AUC=average_precision_score(yt, p),
                Recall=recall_score(yt, yh, zero_division=0), Specificity=tn / (tn + fp), Precision=precision_score(yt, yh, zero_division=0),
                F1=f1_score(yt, yh, zero_division=0), MCC=matthews_corrcoef(yt, yh), Brier=brier_score_loss(yt, p))
inner = StratifiedKFold(3, shuffle=True, random_state=SEED)
def tune(pipe, space, Xtr, ytr, n_iter=N_ITER, seed=SEED):
    sp = {f"clf__estimator__{k}": v for k, v in space.items()}
    rs = RandomizedSearchCV(pipe, sp, n_iter=n_iter, scoring="average_precision", cv=inner, random_state=seed, n_jobs=1, error_score="raise").fit(Xtr, ytr)
    return rs.best_estimator_, rs.best_params_
def youden(yt, p):
    fpr, tpr, th = roc_curve(yt, p); return float(th[np.argmax(tpr - fpr)])
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def _fit(Xd, yy, off=None, it=60):
    b = np.zeros(Xd.shape[1]); off = np.zeros(len(yy)) if off is None else off
    for _ in range(it):
        mu = 1 / (1 + np.exp(-(Xd @ b + off))); W = mu * (1 - mu)
        b = b + np.linalg.solve(Xd.T @ (Xd * W[:, None]) + 1e-9 * np.eye(Xd.shape[1]), Xd.T @ (yy - mu))
    return b
def calib(p, yy):
    lp = logit(p); a = _fit(np.ones((len(yy), 1)), yy, off=lp)[0]; s = _fit(np.c_[np.ones(len(yy)), lp], yy)[1]; return a, s

## 2. Nested CV ulang (menyimpan prediksi per pengulangan dan ambang Youden dari data latih setiap fold)
Perhatian: bagian ini memerlukan waktu lama (±2–4 jam). File yang sudah ada di `out/` dilewati.

In [ ]:
# Ulang nested CV (fold luar & seed identik dengan notebook 01/02), menyimpan prediksi per pengulangan
# dan ambang Youden dari prediksi out-of-fold DALAM data latih tiap fold luar.

outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
splits = list(outer.split(X, y))
combos = [(s, m, ind) for s, ind in [("none", False), ("class_weight", False), ("none", True), ("class_weight", True),
                                      ("smote", False), ("rus", False), ("smotenc", False)] for m in ["LR", "HGB", "RF"]]
t0 = time.time()
for strat, name, ind in combos:
    tag = f"{strat}{'_ind' if ind else ''}_{name}"
    fn = os.path.join(OUTD, f"pred_{tag}.npz")
    if os.path.exists(fn): continue
    est, space = models(cw=(strat == "class_weight"))[name]; samp = strat if strat in ("smote", "rus", "smotenc") else "none"
    P = np.full((N_REPEATS, len(y)), np.nan); TH = np.zeros(len(splits)); PAR = []
    for f, (tr, te) in enumerate(splits):
        pipe = Pipeline([("prep", make_prep(ind)), ("clf", Resampled(est, samp, SEED))])
        best, bp = tune(pipe, space, X.iloc[tr], y[tr], seed=SEED + f)
        oof = cross_val_predict(clone(best), X.iloc[tr], y[tr], cv=inner, method="predict_proba")[:, 1]
        TH[f] = youden(y[tr], oof); P[f // 5, te] = best.predict_proba(X.iloc[te])[:, 1]; PAR.append({k: str(v) for k, v in bp.items()})
        print(f"[{time.time()-t0:6.0f}s] {tag} fold {f}", flush=True)
    np.savez(fn, P=P, TH=TH, PAR=json.dumps(PAR))
print("SELESAI", time.time() - t0)

## 3. Ringkasan: IK bootstrap (rata-rata per pasien vs per pengulangan), kalibrasi, recall/MCC pada ambang Youden, indikator BMI

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, matthews_corrcoef
from scipy import stats
outer=RepeatedStratifiedKFold(n_splits=5,n_repeats=N_REPEATS,random_state=SEED)
splits=list(outer.split(X,y)); pi=y.mean(); bs0=brier_score_loss(y,np.full(len(y),pi))
rng=np.random.default_rng(42)
def boot(f,yy,pp,B=1000):
    v=[];n=len(yy)
    for _ in range(B):
        ix=rng.integers(0,n,n)
        if yy[ix].sum()==0: continue
        v.append(f(yy[ix],pp[ix]))
    return np.percentile(v,[2.5,97.5])
rows=[];perfold={}
for strat,ind in [("none",False),("class_weight",False),("none",True),("class_weight",True),("smote",False),("rus",False),("smotenc",False)]:
  for m in ["LR","HGB","RF"]:
    tag=f"{strat}{'_ind' if ind else ''}_{m}"; fn=f"{OUTD}/pred_{tag}.npz"
    if not os.path.exists(fn): continue
    z=np.load(fn); P=z["P"]; TH=z["TH"]; pm=P.mean(0)
    # recall/MCC at inner-OOF Youden threshold, per outer fold -> pooled per repeat
    rec=[];spe=[];mcc=[];pf=[]
    for r in range(N_REPEATS):
        yhat=np.zeros(len(y),int)
        for f in range(r*5,r*5+5):
            te=splits[f][1]; yhat[te]=(P[r,te]>=TH[f]).astype(int)
            pf.append(average_precision_score(y[te],P[r,te]))
        tp=((yhat==1)&(y==1)).sum(); tn=((yhat==0)&(y==0)).sum()
        rec.append(tp/y.sum()); spe.append(tn/(y==0).sum()); mcc.append(matthews_corrcoef(y,yhat))
    perfold[tag]=pf
    a,b=calib(pm,y); bs=brier_score_loss(y,pm)
    ci_avg=boot(average_precision_score,y,pm)
    ci_rep=np.mean([boot(average_precision_score,y,P[r],500) for r in range(N_REPEATS)],axis=0)
    rows.append(dict(kombo=tag,ROC=roc_auc_score(y,pm),PR=average_precision_score(y,pm),PR_ci_rata2pasien=f"{ci_avg[0]:.3f}–{ci_avg[1]:.3f}",
        PR_ci_per_ulangan=f"{ci_rep[0]:.3f}–{ci_rep[1]:.3f}",PR_perfold_mean=np.mean(pf),Brier=bs,BSS=1-bs/bs0,intercept=a,slope=b,
        ambang_median=np.median(TH),recall_youden=np.mean(rec),spes_youden=np.mean(spe),MCC_youden=np.mean(mcc)))
R=pd.DataFrame(rows); pd.set_option("display.width",250); print(R.round(3).to_string()); R.to_csv(f"{OUTD}/r04_nested_ringkasan_youden_kalibrasi.csv",index=False)
def nb(d,k=10,r=.25):
    d=np.asarray(d); t=d.mean()/np.sqrt((1/k+r)*d.var(ddof=1)); return d.mean(),t,2*stats.t.sf(abs(t),k-1)
print("\nIndikator BMI kosong vs tanpa indikator (PR-AUC per fold, Nadeau-Bengio):")
for s in ["none","class_weight"]:
  for m in ["LR","HGB","RF"]:
    a,b=f"{s}_{m}",f"{s}_ind_{m}"
    if a in perfold and b in perfold:
        md,t,p=nb(np.array(perfold[b])-np.array(perfold[a])); print(f"  {s:13s}{m:4s} selisih {md:+.4f}  t={t:.2f} p={p:.3f}")

## 4. Selisih PR-AUC per fold (optimasi − bawaan), uji primer Nadeau–Bengio + Holm, stabilitas peringkat

In [ ]:
import pandas as pd, numpy as np, json
from scipy import stats
H = HASIL + "/"
d=pd.concat([pd.read_csv(H+"eksp2_per_fold.csv"),pd.read_csv(H+"eksp4_smotenc_per_fold.csv")])
print(d.Mode.unique(), d.Strategi.unique())
def nb(diff,k=10,r=0.25):
    diff=np.asarray(diff); m=diff.mean(); v=diff.var(ddof=1)
    t=m/np.sqrt((1/k+r)*v) if v>0 else np.inf; return t,2*stats.t.sf(abs(t),k-1)
rows=[]
for (s,mo),g in d.groupby(["Strategi","Model"]):
    a=g[g.Mode=="default"].sort_values("Fold").PR_AUC.values; b=g[g.Mode!="default"].sort_values("Fold").PR_AUC.values
    df=b-a; t,p=nb(df)
    rows.append(dict(Strategi=s,Model=mo,PR_bawaan_mean=a.mean(),PR_opt_mean=b.mean(),selisih_mean=df.mean(),selisih_min=df.min(),selisih_max=df.max(),
                     n_naik=int((df>0).sum()),NB_t=t,NB_p=p))
r=pd.DataFrame(rows); r["NB_p_Holm_15"]=np.minimum(1,np.sort(r.NB_p.values)[np.argsort(np.argsort(r.NB_p.values))]*0) # placeholder
# Holm over 15
o=np.argsort(r.NB_p.values); ph=np.empty(15); run=0
for i,ix in enumerate(o): run=max(run,(15-i)*r.NB_p.values[ix]); ph[ix]=min(1,run)
r["NB_p_Holm_15"]=ph
print(r.round(3).to_string()); r.to_csv(OUTD + "/r04_selisih_per_fold_tuning.csv",index=False)
# ranking stability: tuned mode, per fold PR-AUC; bootstrap folds
t=d[d.Mode!="default"].copy(); t["kombo"]=t.Strategi+"_"+t.Model
W=t.pivot_table(index="Fold",columns="kombo",values="PR_AUC")
rng=np.random.default_rng(42); win={c:0 for c in W.columns}; top3={c:0 for c in W.columns}
for _ in range(2000):
    ix=rng.integers(0,10,10); m=W.iloc[ix].mean().sort_values(ascending=False)
    win[m.index[0]]+=1
    for c in m.index[:3]: top3[c]+=1
st=pd.DataFrame(dict(mean=W.mean(),juara=pd.Series(win)/2000,top3=pd.Series(top3)/2000)).sort_values("mean",ascending=False)
print(st.round(3)); st.to_csv(OUTD + "/r04_stabilitas_peringkat.csv")

## 5. Analisis tambahan lain

In [ ]:
from sklearn.inspection import permutation_importance
R = {}
Xb = X.drop(columns="bmi_kosong")
# ---------- 1. BMI kosong ----------
m = X.bmi_kosong == 1
R["bmi"] = dict(n_kosong=int(m.sum()), stroke_kosong=int(y[m].sum()), laju_kosong=float(y[m].mean()), n_ada=int((~m).sum()),
               stroke_ada=int(y[~m].sum()), laju_ada=float(y[~m].mean()), usia_median_kosong=float(X.age[m].median()), usia_median_ada=float(X.age[~m].median()))
from scipy.stats import fisher_exact
R["bmi"]["odds_ratio"], R["bmi"]["p_fisher"] = [float(v) for v in fisher_exact([[y[m].sum(), (1 - y[m]).sum()], [y[~m].sum(), (1 - y[~m]).sum()]])]
# ---------- 2. Skenario bocor: SMOTE & ROS sebelum split, evaluasi juga pada data uji ASLI ----------
allm = {k: v[0] for k, v in models(False).items()}; allm["KNN"] = KNeighborsClassifier(5)
Xp = make_prep().fit_transform(X); rows = []
for samp in ["smote", "ros"]:
    Xs, ys = (smote if samp == "smote" else ros)(Xp, y, seed=SEED)
    syn = np.r_[np.zeros(len(y), bool), np.ones(len(ys) - len(y), bool)]
    itr, ite = train_test_split(np.arange(len(ys)), test_size=0.2, stratify=ys, random_state=SEED)
    real_te = ite[~syn[ite]]
    for name, est in allm.items():
        mdl = clone(est).fit(Xs[itr], ys[itr]); p_all = mdl.predict_proba(Xs[ite])[:, 1]; p_real = mdl.predict_proba(Xs[real_te])[:, 1]
        for lab, yt, p in [("uji campuran (asli+sintetis)", ys[ite], p_all), ("uji hanya data asli", ys[real_te], p_real)]:
            mm = metrics(yt, p); rows.append(dict(Skenario=f"{samp.upper()} sebelum split 80/20", Evaluasi=lab, Model=name, n_uji=len(yt), positif=int(yt.sum()),
                                                prevalensi=yt.mean(), **mm, PR_AUC_lift=mm["PR_AUC"] / yt.mean()))
    # sebelum 5-fold CV
    for name, est in allm.items():
        P = cross_val_predict(clone(est), Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
        for lab, msk in [("uji campuran (asli+sintetis)", np.ones(len(ys), bool)), ("uji hanya data asli", ~syn)]:
            mm = metrics(ys[msk], P[msk]); rows.append(dict(Skenario=f"{samp.upper()} sebelum 5-fold CV", Evaluasi=lab, Model=name, n_uji=int(msk.sum()), positif=int(ys[msk].sum()),
                                                          prevalensi=ys[msk].mean(), **mm, PR_AUC_lift=mm["PR_AUC"] / ys[msk].mean()))
    if samp == "ros":
        tr_rows = {tuple(r) for r in np.round(Xs[itr][ys[itr] == 1], 8)}
        pos_te = ite[ys[ite] == 1]
        R["ros"] = dict(test_positif=int(len(pos_te)), test_positif_salinan=int(syn[pos_te].sum()),
                        test_positif_identik_dgn_latih=int(sum(tuple(r) in tr_rows for r in np.round(Xs[pos_te], 8))))
# skenario benar (pembanding, prevalensi asli)
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
for name, est in allm.items():
    p = Pipeline([("prep", make_prep()), ("clf", Resampled(est, "smote", SEED))]).fit(xtr, ytr).predict_proba(xte)[:, 1]
    mm = metrics(yte, p); rows.append(dict(Skenario="Benar: split dulu, SMOTE di data latih", Evaluasi="uji asli", Model=name, n_uji=len(yte), positif=int(yte.sum()),
                                          prevalensi=yte.mean(), **mm, PR_AUC_lift=mm["PR_AUC"] / yte.mean()))
e1 = pd.DataFrame(rows); e1.to_csv(os.path.join(OUTD, "r04_skenario_prevalensi_ros.csv"), index=False)
# ---------- 3. Jarak (definisi eksplisit) ----------
Xs, ys = smote(Xp, y, seed=SEED); syn = np.r_[np.zeros(len(y), bool), np.ones(len(ys) - len(y), bool)]
itr, ite = train_test_split(np.arange(len(ys)), test_size=0.2, stratify=ys, random_state=SEED); pos_te = ite[ys[ite] == 1]
nnTr = NearestNeighbors(n_neighbors=1).fit(Xs[itr][ys[itr] == 1])
d_syn = nnTr.kneighbors(Xs[pos_te][syn[pos_te]])[0].ravel(); d_realte = nnTr.kneighbors(Xs[pos_te][~syn[pos_te]])[0].ravel()
d_loo = NearestNeighbors(n_neighbors=2).fit(Xp[y == 1]).kneighbors(Xp[y == 1])[0][:, 1]
# jarak tetangga terdekat pasien stroke asli ke pasien stroke asli lain DI DATA LATIH skenario A (pembanding paling setara)
real_tr_pos = itr[(ys[itr] == 1) & (~syn[itr])]
d_real_to_realtr = NearestNeighbors(n_neighbors=1).fit(Xs[real_tr_pos]).kneighbors(Xs[pos_te][~syn[pos_te]])[0].ravel()
R["jarak"] = dict(ruang="fitur hasil praproses: 3 numerik terstandardisasi + 2 biner + 10 kolom one-hot (15 dimensi), jarak Euclidean",
    n_uji_positif=int(len(pos_te)), n_sintetis=int(syn[pos_te].sum()), n_asli=int((~syn[pos_te]).sum()),
    median_NN_sintetis_uji_ke_positif_latih=float(np.median(d_syn)), median_NN_asli_uji_ke_positif_latih=float(np.median(d_realte)),
    median_NN_LOO_antar_stroke_asli=float(np.median(d_loo)), median_NN_asli_uji_ke_stroke_asli_latih=float(np.median(d_real_to_realtr)))
R["jarak"]["rasio_LOO_vs_sintetis"] = R["jarak"]["median_NN_LOO_antar_stroke_asli"] / R["jarak"]["median_NN_sintetis_uji_ke_positif_latih"]
# numerik saja
Xn = Xp[:, :3]; Xsn = Xs[:, :3]
nnn = NearestNeighbors(n_neighbors=1).fit(Xsn[itr][ys[itr] == 1])
R["jarak"]["numerik_saja_median_sintetis"] = float(np.median(nnn.kneighbors(Xsn[pos_te][syn[pos_te]])[0]))
R["jarak"]["numerik_saja_median_LOO_asli"] = float(np.median(NearestNeighbors(n_neighbors=2).fit(Xn[y == 1]).kneighbors(Xn[y == 1])[0][:, 1]))
# ---------- 4. Variasi antar pembagian 80:20 (skenario benar, 20 seed) ----------
rows = []
for s in range(20):
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=s)
    for name in ["LR", "RF", "HGB"]:
        p = Pipeline([("prep", make_prep()), ("clf", Resampled(allm[name], "smote", SEED))]).fit(a, c).predict_proba(b)[:, 1]
        rows.append(dict(seed=s, Model=name, PR_AUC=average_precision_score(d, p), ROC_AUC=roc_auc_score(d, p)))
sv = pd.DataFrame(rows); sv.to_csv(os.path.join(OUTD, "r04_variasi_seed_skenarioB.csv"), index=False)
# ---------- 5. Baseline usia saja ----------
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
age = X[["age"]].values; P = np.zeros(len(y)); cnt = np.zeros(len(y)); fpr = []
for f, (tr, te) in enumerate(outer.split(X, y)):
    mdl = LogisticRegression(max_iter=3000).fit(age[tr], y[tr]); p = mdl.predict_proba(age[te])[:, 1]; P[te] += p; cnt[te] += 1
    fpr.append(dict(Fold=f, PR_AUC=average_precision_score(y[te], p), ROC_AUC=roc_auc_score(y[te], p)))
P /= cnt; mm = metrics(y, P)
mdl = LogisticRegression(max_iter=3000).fit(xtr[["age"]].values, ytr); pa = mdl.predict_proba(xte[["age"]].values)[:, 1]
oof = cross_val_predict(LogisticRegression(max_iter=3000), xtr[["age"]].values, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
th = youden(ytr, oof); mt = metrics(yte, pa, th); a_, s_ = calib(pa, yte)
R["usia_saja"] = dict(nestedcv=mm, per_fold=pd.DataFrame(fpr).mean().to_dict(), uji=metrics(yte, pa), uji_youden=dict(ambang=th, **mt), uji_intercept=a_, uji_slope=s_)
pd.DataFrame(fpr).to_csv(os.path.join(OUTD, "r04_usia_saja_per_fold.csv"), index=False)
# ---------- 6. Model akhir: reproduksi, kandidat 12, indikator BMI, PI OOF, gambar kalibrasi ----------
hp = json.load(open(os.path.join(HASIL, "eksp3b_best_params.json")))
conv = lambda v: None if v == "None" else (float(v) if "." in v else int(v))
kw = {k.split("__")[-1]: conv(v) for k, v in hp.items()}
def final(ind=False): return Pipeline([("prep", make_prep(ind)), ("clf", Resampled(HistGradientBoostingClassifier(random_state=SEED, **kw), "none", SEED))])
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED); fin = {}
for ind in [False, True]:
    raw = final(ind).fit(xtr, ytr); p = raw.predict_proba(xte)[:, 1]
    oof = cross_val_predict(final(ind), xtr, ytr, cv=cv5, method="predict_proba")[:, 1]; th = youden(ytr, oof)
    iso = __import__("sklearn.isotonic", fromlist=["x"]).IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof, ytr)
    fin[ind] = dict(p=p, oof=oof, th=th, iso=iso)
    a_, s_ = calib(p, yte); R[f"akhir_indikator_{ind}"] = dict(ambang=th, **metrics(yte, p, th), intercept=a_, slope=s_, ROC_05=None)
# 12 kandidat (sama dengan nested CV)
est, space = models(False)["HGB"]
b12, bp12 = tune(Pipeline([("prep", make_prep()), ("clf", Resampled(est, "none", SEED))]), space, xtr, ytr, n_iter=12)
p12 = b12.predict_proba(xte)[:, 1]; oof12 = cross_val_predict(clone(b12), xtr, ytr, cv=cv5, method="predict_proba")[:, 1]; th12 = youden(ytr, oof12)
R["akhir_12_kandidat"] = dict(params={k: str(v) for k, v in bp12.items()}, ambang=th12, **metrics(yte, p12, th12))
# Indikator BMI dengan tuning ulang (24 kandidat, sama dengan model akhir)
bI, bpI = tune(Pipeline([("prep", make_prep(True)), ("clf", Resampled(est, "none", SEED))]), space, xtr, ytr, n_iter=24)
pI = bI.predict_proba(xte)[:, 1]; oofI = cross_val_predict(clone(bI), xtr, ytr, cv=cv5, method="predict_proba")[:, 1]; thI = youden(ytr, oofI)
R["akhir_indikator_tuning_ulang"] = dict(params={k: str(v) for k, v in bpI.items()}, ambang=thI, **metrics(yte, pI, thI), intercept=calib(pI, yte)[0], slope=calib(pI, yte)[1])
# Permutation importance: data uji vs OOF data latih (5-fold, 20 ulangan)
cols = list(Xb.columns)
pi_te = permutation_importance(final().fit(xtr, ytr), xte, yte, scoring="average_precision", n_repeats=20, random_state=SEED)
rows = []
for f, (tr, va) in enumerate(cv5.split(xtr, ytr)):
    mdl = final().fit(xtr.iloc[tr], ytr[tr]); pi = permutation_importance(mdl, xtr.iloc[va], ytr[va], scoring="average_precision", n_repeats=20, random_state=SEED)
    for c, mu in zip(xtr.columns, pi.importances_mean): rows.append(dict(Fold=f, Fitur=c, imp=mu))
po = pd.DataFrame(rows); po = po[po.Fitur.isin(cols)]
pit = pd.DataFrame(dict(Fitur=xte.columns, uji_mean=pi_te.importances_mean, uji_sd=pi_te.importances_std)); pit = pit[pit.Fitur.isin(cols)]
pim = po.groupby("Fitur").imp.agg(oof_mean="mean", oof_min="min", oof_max="max").reset_index().merge(pit, on="Fitur").sort_values("oof_mean", ascending=False)
pim.to_csv(os.path.join(OUTD, "r04_permutation_importance_oof_vs_uji.csv"), index=False)
np.savez(os.path.join(OUTD, "pred_akhir.npz"), p=fin[False]["p"], pcal=fin[False]["iso"].predict(fin[False]["p"]), yte=yte, oof=fin[False]["oof"], ytr=ytr)
e1r = e1.round(4); json.dump(R, open(os.path.join(OUTD, "r04_ringkasan.json"), "w"), indent=1, default=float)
print(json.dumps(R, indent=1, default=lambda v: round(float(v), 4) if isinstance(v, (float, np.floating)) else str(v))[:6000])
print(e1[["Skenario", "Evaluasi", "Model", "n_uji", "positif", "Accuracy", "ROC_AUC", "PR_AUC", "PR_AUC_lift", "Recall", "MCC"]].round(3).to_string())
print(sv.groupby("Model")[["PR_AUC", "ROC_AUC"]].describe().round(3).to_string()); print(sv[sv.seed == 42] if 42 in sv.seed.values else "")
print(pim.round(4).to_string())

## 6. Gambar kalibrasi revisi (rentang 0–0,3, loess, histogram)

In [ ]:
import numpy as np, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
d=np.load(os.path.join(OUTD, "pred_akhir.npz")); y=d["yte"].astype(float); p=d["p"]; pc=d["pcal"]
lg=lambda v: np.log(v/(1-v))
def lowess(x,yy,grid,frac=0.75):
    k=int(np.ceil(frac*len(x))); out=np.empty(len(grid))
    for i,g in enumerate(grid):
        dist=np.abs(x-g); h=np.partition(dist,k-1)[k-1]+1e-12
        w=np.clip(1-(dist/h)**3,0,None)**3; W=w.sum(); xm=(w*x).sum()/W; ym=(w*yy).sum()/W
        sxx=(w*(x-xm)**2).sum(); b=(w*(x-xm)*(yy-ym)).sum()/sxx if sxx>1e-9 else 0
        out[i]=ym+b*(g-xm)
    return np.clip(out,0,1)
def wilson(k,n,z=1.96):
    ph=k/n; den=1+z*z/n; c=(ph+z*z/(2*n))/den; h=z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den; return c-h,c+h
rng=np.random.default_rng(42)
fig,(ax,ah)=plt.subplots(2,1,figsize=(4.6,5.4),gridspec_kw=dict(height_ratios=[3,1.1]),sharex=True)
ax.plot([0,.3],[0,.3],"k--",lw=.8,label="Kalibrasi sempurna")
x=lg(p); grid=np.linspace(np.quantile(p,.01),np.quantile(p,.99),60); gx=lg(grid)
f=lowess(x,y,gx); B=[]
for _ in range(500):
    ix=rng.integers(0,len(y),len(y)); B.append(lowess(x[ix],y[ix],gx))
lo,up=np.percentile(B,[2.5,97.5],axis=0)
ax.fill_between(grid,lo,up,color="#6b6b6b",alpha=.18,lw=0)
ax.plot(grid,f,color="#4d4d4d",lw=1.8,label="Tanpa kalibrasi (loess, IK 95% bootstrap)")
u=np.unique(pc); first=True
for v in u:
    m=pc==v; n=m.sum(); k=y[m].sum()
    if n<5: continue
    a,b=wilson(k,n)
    ax.errorbar(v,k/n,yerr=[[k/n-a],[b-k/n]],fmt="o",color="#1f5fa8",ms=3+2*np.log10(n),mfc="white",capsize=2,lw=1,
                label="Kalibrasi isotonik (per nilai, IK 95% Wilson)" if first else None); first=False
ax.set_xlim(0,.3); ax.set_ylim(0,.3); ax.set_ylabel("Proporsi stroke teramati"); ax.legend(fontsize=7,loc="lower right",frameon=False)
bins=np.linspace(0,.3,31)
ah.hist(p[y==0],bins=bins,color="#bbbbbb",label="Bukan stroke"); ah.hist(p[y==1],bins=bins,color="#c0392b",label="Stroke")
ah.set_yscale("log"); ah.set_xlabel("Probabilitas prediksi"); ah.set_ylabel("Jumlah (log)"); ah.legend(fontsize=7,frameon=False)
ah.text(.995,.05,"Histogram: prediksi tanpa kalibrasi",transform=ah.transAxes,ha="right",fontsize=6.5,color="#555")
fig.tight_layout(); fig.savefig(os.path.join(OUTD, "gambar2_kalibrasi_revisi.png"),dpi=300)

## 7. Verifikasi SMOTE terhadap imbalanced-learn

In [ ]:
# Verifikasi implementasi SMOTE terhadap imbalanced-learn (komentar reviewer #28)
# Jalankan setelah: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE as IMB_SMOTE
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
class ImbSMOTE(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator=None, random_state=0): self.estimator = estimator; self.random_state = random_state
    def fit(self, X_, y_):
        Xr, yr = IMB_SMOTE(k_neighbors=5, random_state=self.random_state).fit_resample(np.asarray(X_, float), np.asarray(y_))
        self.est_ = clone(self.estimator).fit(Xr, yr); self.classes_ = self.est_.classes_; return self
    def predict_proba(self, X_): return self.est_.predict_proba(np.asarray(X_, float))
hasil = []
for name, (est, _) in models().items():
    for lab, clf in [("implementasi sendiri", Resampled(est, "smote", SEED)), ("imbalanced-learn", ImbSMOTE(est, SEED))]:
        p = Pipeline([("prep", make_prep()), ("clf", clf)]).fit(xtr, ytr).predict_proba(xte)[:, 1]
        hasil.append(dict(Model=name, SMOTE=lab, **{k: round(v, 3) for k, v in metrics(yte, p).items()}))
    # validasi silang 5-fold di data latih (lebih stabil daripada satu data uji)
    for lab, clf in [("implementasi sendiri", Resampled(est, "smote", SEED)), ("imbalanced-learn", ImbSMOTE(est, SEED))]:
        oof = cross_val_predict(Pipeline([("prep", make_prep()), ("clf", clf)]), xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
        hasil.append(dict(Model=name, SMOTE=lab + " (CV 5-fold data latih)", ROC_AUC=round(roc_auc_score(ytr, oof), 3), PR_AUC=round(average_precision_score(ytr, oof), 3)))
v = pd.DataFrame(hasil); v.to_csv(os.path.join(OUTD, "r04_verifikasi_imblearn.csv"), index=False); v